<a href="https://colab.research.google.com/github/markajbell/BH/blob/main/Stats10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [36]:
import pandas as pd
import json
import os
import glob

In [37]:
pd.set_option('display.max_rows', None)
pd.set_option("display.max_rows", None, "display.max_columns", None)
pd.set_option('display.max_colwidth', None)

In [38]:
!wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/adsync/secure.json
#!wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/adsync/secure10k.json

#!mkdir = "/content/adsync"
!mv secure.json /content/adsync
#!mv secure10k.json /content/adsync
#!mv vul_50k.json /content/adsync
path = "/content/adsync"
data = {}
json_files = glob.glob(os.path.join(path, "*.json"))
for file in json_files:
    filename = os.path.basename(file).replace(".json", "")
    suf = filename.split("_")[-1]

    with open(file, 'r', encoding='utf-8') as f:
        content = json.load(f)

    if "data" in content:
        if suf in data:
            data[f"{suf}"].extend(content["data"])
            #print(f" The key 'data' was found in {f}.")
        else:
            data[f"{suf}"] = content["data"]
            #print(f" The key 'data' was found in {f}.")
    else:
        print(f"Warning: The key 'data' was not found in {f}.")

--2025-08-01 16:21:44--  https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/adsync/secure.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1314149 (1.3M) [text/plain]
Saving to: ‘secure.json’

secure.json         100%[===================>]   1.25M  --.-KB/s    in 0.01s   

2025-08-01 16:21:44 (124 MB/s) - ‘secure.json’ saved [1314149/1314149]



In [39]:
# prompt: if the type column = "node" save that record into a pandas dataframe called nodes, if the type = "relationship' then save to a pandas dataframe called edges

import pandas as pd
df = pd.DataFrame(data[suf])
nodes = df[df['type'] == 'node'].copy()
edges = df[df['type'] == 'relationship'].copy()
nodes['id'] = nodes['id'].astype(int)

In [40]:
# prompt: for every record in "nodes" create a l1, l2 and l3 column for the values in "nodes.labels" eg. for record 1 l1=Base, l2=OU and l3 would be null

import pandas as pd
def extract_labels(labels):
    # Ensure labels is a list
    if not isinstance(labels, list):
        return [None, None, None]

    l1 = labels[0] if len(labels) > 0 else None
    l2 = labels[1] if len(labels) > 1 else None
    l3 = labels[2] if len(labels) > 2 else None
    return [l1, l2, l3]

# Apply the function to create the new columns
nodes[['l1', 'l2', 'l3']] = nodes['labels'].apply(lambda x: pd.Series(extract_labels(x)))

In [41]:
nodes = nodes.drop(columns=['start', 'end', 'label', 'labels'])
edges = edges.drop(columns=['labels'])

In [42]:
#edges = edges.drop(columns=['id'])
import numpy as np
edges['id'] = np.arange(len(edges)).astype('int64')
nodes['weight'] = 1

In [43]:
lastnode=(len(str(nodes['id'].iloc[-1])))

new_node_id = int('9' * lastnode)

print(f"The new node ID is: {new_node_id}")

The new node ID is: 99999


In [44]:
#prompt: create a new record in nodes where id=new_node_id, properties = {'domain': 'TESTLAB.LOCALE', 'name': 'ALL_T0@TESTLAB.LOCALE', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-99999', 'distinguishedname': 'CN=ALL_T0,OU=ALL_T0,OU=ALL_T0,DC=TESTLAB,DC=LOCALE', 'description': 'All Tier 0 nodes', 'highvalue': True, 'admincount': True, 'owned': False}, type=node, l1=ALL_T0, l2=ALL_T0, l3=ALL_T0

# Create a dictionary with the new record's data
new_node_data = {
    'id': new_node_id,
    'properties': {'domain': 'TESTLAB.LOCALE', 'name': 'ALL_T0@TESTLAB.LOCALE', 'objectid': 'S-1-5-21-883232822-274137685-4173207997-' + str(new_node_id), 'distinguishedname': 'CN=ALL_T0,OU=ALL_T0,OU=ALL_T0,DC=TESTLAB,DC=LOCALE', 'description': 'All Tier 0 nodes', 'highvalue': True, 'admincount': True, 'owned': False},
    'type': 'node',
    'l1': 'ALL_T0',
    'l2': 'ALL_T0',
    'l3': 'ALL_T0',
    'weight': int(100) # Assuming a default weight, you can adjust if needed
}
#new_node_data

# Convert the dictionary to a DataFrame
new_node_df = pd.DataFrame([new_node_data])

# Concatenate the new DataFrame with the existing nodes DataFrame
nodes = pd.concat([nodes, new_node_df], ignore_index=True)

# Display the last few rows to confirm the addition
print("Nodes DataFrame after adding the new record:")
#display(nodes.tail())

Nodes DataFrame after adding the new record:


In [45]:
#1/0
# tier0_node_ids = nodes[nodes['properties'].apply(lambda x: 'Tier 0' in x.get('distinguishedname', ''))]['id'].tolist()
# tier0_node_ids

tier0_node_ids = nodes[
    nodes['properties'].apply(
        lambda x: any(
            term in x.get('distinguishedname', '')
            for term in ['TIER 0', 'Tier 0', 'T0', 'TIER0']
        )
    )
]['id'].tolist()


tier1_node_ids = nodes[
    nodes['properties'].apply(
        lambda x: any(
            term in x.get('distinguishedname', '')
            for term in ['TIER 1', 'Tier 1', 'T1']
        )
    )
]['id'].tolist()

tier2_node_ids = nodes[
    nodes['properties'].apply(
        lambda x: any(
            term in x.get('distinguishedname', '')
            for term in ['TIER 2', 'Tier 2', 'T2']
        )
    )
]['id'].tolist()

print(f"tier 0 ",len(tier0_node_ids))
# tier1_node_ids = nodes[nodes['properties'].apply(lambda x: 'Tier 1' in x.get('distinguishedname', ''))]['id'].tolist()
print(f"tier 1 ",len(tier1_node_ids))

# tier2_node_ids = nodes[nodes['properties'].apply(lambda x: 'Tier 2' in x.get('distinguishedname', ''))]['id'].tolist()
print(f"tier 2 ",len(tier2_node_ids))

tier 0  518
tier 1  701
tier 2  8933


In [46]:
# prompt: find records from nodes where "objectid" ends with match(r'.*-\d{3}$' and highvalue=true and admincount=true.

import re

# Filter nodes where 'objectid' ends with a dash followed by three digits between 500 and 599.
filtered_nodes = nodes[
    nodes['properties'].apply(
        lambda x: re.search(r'.*-5[0-9]{2}$', x.get('objectid', '')) is not None and
                  x.get('highvalue', False) == True and
                  x.get('admincount', False) == True
    )
]

# Display the filtered nodes
print("Records where 'objectid' ends with - followed by three digits, and 'highvalue' and 'admincount' are True:")
len(filtered_nodes)

Records where 'objectid' ends with - followed by three digits, and 'highvalue' and 'admincount' are True:


8

In [47]:
# prompt:  check if "id" from filtered_nodes is in tier0_node_ids, if not then append to tier0_node_ids

# Assuming filtered_nodes DataFrame is already defined from the preceding code

# Iterate through the 'id' column of the filtered_nodes DataFrame
for node_id in filtered_nodes['id']:
    # Check if the node_id is not already in tier0_node_ids
    if node_id not in tier0_node_ids:
        # Append the node_id to the tier0_node_ids list
        tier0_node_ids.append(node_id)

# Print the updated length and contents of tier0_node_ids
print(f"tier 0 (after checking filtered_nodes): {len(tier0_node_ids)}")
print("Updated Tier 0 Node IDs:", tier0_node_ids)


tier 0 (after checking filtered_nodes): 526
Updated Tier 0 Node IDs: [6225, 6228, 6229, 6230, 6231, 6373, 6374, 6375, 6376, 6377, 6378, 6381, 6387, 6389, 6390, 6401, 6406, 6409, 6412, 6416, 6424, 6425, 6429, 6430, 6431, 6433, 6434, 6437, 6438, 6445, 6446, 6447, 6450, 6452, 6458, 6459, 6460, 6461, 6462, 6463, 6465, 6467, 6469, 6471, 6476, 6478, 6482, 6483, 6484, 6485, 6490, 6497, 6502, 6503, 6509, 6510, 6511, 6512, 6515, 6520, 6529, 6530, 6534, 6536, 6538, 6540, 6542, 6544, 6546, 6550, 6551, 6559, 6563, 6564, 6568, 6572, 6574, 6576, 6578, 6579, 6587, 6593, 6595, 6597, 6603, 6604, 6608, 6609, 6610, 6612, 6615, 6616, 6622, 6628, 6633, 6635, 6640, 6643, 6651, 6652, 6660, 6664, 6671, 6674, 6678, 6679, 6684, 6686, 6688, 6689, 6697, 6698, 6699, 6704, 6738, 6781, 6800, 6801, 6824, 6826, 6830, 6838, 6858, 6862, 6902, 6956, 6978, 6982, 7003, 7009, 7012, 7076, 7094, 7104, 7144, 7157, 7167, 7188, 7210, 7233, 7237, 7255, 7278, 7282, 7295, 7301, 7329, 7331, 7335, 7341, 7345, 7346, 7373, 7381, 7415, 

In [48]:
# Iterate through nodes and add a 'Tier0' column
nodes['Tier0'] = 0  # Initialize 'Tier0' column with 0
nodes['Tier1'] = 0  # Initialize 'Tier0' column with 0
nodes['Tier2'] = 0  # Initialize 'Tier0' column with 0

# Set 'Tier0' to 1 if node id is in tier0_node_ids
nodes.loc[nodes['id'].isin(tier0_node_ids), 'Tier0'] = 1

# Set 'Tier1' to 1 if node id is in tier1_node_ids
nodes.loc[nodes['id'].isin(tier1_node_ids), 'Tier1'] = 1

# Set 'Tier2' to 1 if node id is in tier2_node_ids
nodes.loc[nodes['id'].isin(tier2_node_ids), 'Tier2'] = 1

# Display the updated nodes DataFrame (optional)
# display(nodes.head())
# display(nodes.tail())

In [49]:
# nodes.loc[nodes['id'].isin(tier0_node_ids), 'Tier0'] = 1
# nodes.loc[nodes['id'].isin(tier1_node_ids), 'Tier1'] = 1
# nodes.loc[nodes['id'].isin(tier2_node_ids), 'Tier2'] = 1

# Show records where Tier0 is 1 and Tier2 is 1
nodes_tier0_and_tier2 = nodes[(nodes['Tier0'] == 1) & (nodes['Tier1'] == 1)]
display(nodes_tier0_and_tier2['id'])
ids_to_update = nodes_tier0_and_tier2['id'].tolist()

# Set the 'Tier0' column to 0 for the rows where the 'id' is in ids_to_update
nodes.loc[nodes['id'].isin(ids_to_update), 'Tier0'] = 0
ids_to_update = 0
nodes_tier0_and_tier2 = nodes[(nodes['Tier0'] == 1) & (nodes['Tier2'] == 1)]
display(nodes_tier0_and_tier2['id'])
ids_to_update = nodes_tier0_and_tier2['id'].tolist()

# Set the 'Tier0' column to 0 for the rows where the 'id' is in ids_to_update
nodes.loc[nodes['id'].isin(ids_to_update), 'Tier0'] = 0


tier0 = []
for index, row in nodes.iterrows():
    node_id = row['id']
    Tier2 = row['Tier2']
    if node_id in tier0_node_ids:
        if Tier2 == 1:
            tier0.append(node_id)


tier0_node_ids = [id for id in tier0_node_ids if id not in tier0]

tier1 = []
for index, row in nodes.iterrows():
    node_id = row['id']
    Tier1 = row['Tier1']
    if node_id in tier0_node_ids:
        if Tier1 == 1:
            tier1.append(node_id)


tier0_node_ids = [id for id in tier0_node_ids if id not in tier1]


len(tier0_node_ids)

,id
239,6460
263,6484
383,6604
401,6622
431,6652


,id
212,6433
231,6452
237,6458
313,6534
463,6684
465,6686
483,6704
517,6738
560,6781
579,6800


343

In [71]:
import pandas as pd

len(tier0_node_ids)
tier0_node_ids = pd.DataFrame(tier0_node_ids)

In [72]:

for value in tier0_node_ids:
    if value  == new_node_id:
        nodes.loc[nodes['id'] == value, 'weight'] = 100
    else:
        nodes.loc[nodes['id'] == value, 'weight'] = 20

for value in tier1_node_ids:
    nodes.loc[nodes['id'] == value, 'weight'] = 10

for value in tier2_node_ids:
    nodes.loc[nodes['id'] == value, 'weight'] = 5


In [52]:
edges.tail()

,id,properties,type,start,end,label
58391,48316,"{'isacl': True, 'isInherited': True, 'inheritanceType': 'All'}",relationship,"{'id': '6290', 'labels': ['Base', 'Group']}","{'id': '6224', 'labels': ['Base', 'OU']}",GenericAll
58392,48317,"{'isacl': True, 'isInherited': False, 'inheritanceType': 'All'}",relationship,"{'id': '6291', 'labels': ['Base', 'Group']}","{'id': '6222', 'labels': ['Base', 'OU']}",GenericAll
58393,48318,"{'isacl': True, 'isInherited': False, 'inheritanceType': 'All'}",relationship,"{'id': '6291', 'labels': ['Base', 'Group']}","{'id': '6224', 'labels': ['Base', 'OU']}",GenericAll
58394,48319,"{'isacl': True, 'isInherited': False, 'inheritanceType': 'All'}",relationship,"{'id': '6285', 'labels': ['Base', 'Group']}","{'id': '6326', 'labels': ['Base', 'OU']}",GenericAll
58395,48320,"{'isacl': True, 'isInherited': False, 'inheritanceType': 'All'}",relationship,"{'id': '6291', 'labels': ['Base', 'Group']}","{'id': '6326', 'labels': ['Base', 'OU']}",GenericAll


In [53]:
edges['start'] = edges['start'].apply(lambda x: x.get('id') if isinstance(x, dict) else x)
edges['end'] = edges['end'].apply(lambda x: x.get('id') if isinstance(x, dict) else x)
edges = edges.drop(columns=['properties']) #, 'id'])
edges.head()

,id,type,start,end,label
10075,0,relationship,6221,6222,Contains
10076,1,relationship,6221,6223,Contains
10077,2,relationship,6221,6224,Contains
10078,3,relationship,6222,6225,Contains
10079,4,relationship,6222,6226,Contains


In [ ]:
# prompt: for each value in tier0_node_ids add a new record in edges where start = value and end = new_node_id, label=TIER0,weight=100 id=id+1

# Get the maximum existing edge ID
max_edge_id = edges['id'].max()
max_edge_id
new_edges = []
for i, tier0_id in enumerate(tier0_node_ids):
    new_edge = {
        'type': 'relationship',  # Assuming type is always 'relationship' for edges
        'start': tier0_id,
        'end': new_node_id,
        'label': 'TIER0',
        'weight': 100,
        'id': max_edge_id + i + 1 # Assign a unique ID
    }
    new_edges.append(new_edge)

# Convert the list of new edges to a DataFrame
new_edges_df = pd.DataFrame(new_edges)

# Concatenate the new edges DataFrame with the existing edges DataFrame
edges = pd.concat([edges, new_edges_df], ignore_index=True)

# Display the last few rows to confirm the addition
print("Edges DataFrame after adding new TIER0 edges:")
display(edges.tail(344))

In [55]:
# Apply the weights based on the 'label' column

def assign_weight(label):
    if label == "GenericAll":
        return 9
    elif label == "Owns":
        return 10
    elif label == "GenericWrite":
        return 2
    elif label == "AllExtendedRights":
        return 6
    elif label == "CanRDP":
        return 2
    elif label == "Contains":
        return 2
    elif label == "DCSync":
        return 8
    elif label == "WriteDacl":
        return 5
    elif label == "WriteOwner":
        return 7
    elif label == "AddKeyCredentialLink":
        return 6
    elif label == "AdminTo":
        return 8
    elif label == "MemberOf":
        return 1
    elif label == "CanPSRemote":
        return 2
    elif label == "ExecuteDCOM":
        return 2
    elif label == "GPLink":
        return 3
    elif label == "HasSession":
        return 9
    elif label == "ReadLAPSPassword":
        return 2
    elif label == "GetChanges":
        return 3
    elif label == "GetChangesAll":
        return 3
    elif label == "AddSelf":
        return 3
    elif label == "ForceChangePassword":
        return 3
    elif label == "AddMember":
        return 5
    elif label == "AllowedToDelegate":
        return 5
    elif label == "AllowedToAct":
        return 4
    elif label == "TIER0":
        return 100
    else:
        return 1 # Default weight for other labels

edges['weight'] = edges['label'].apply(assign_weight)

# Display the updated edges_all DataFrame with the 'weight' column
print("\nEdges_all DataFrame with weight column:")
edges.head()
#clear_output()


Edges_all DataFrame with weight column:


,id,type,start,end,label,weight
0,0,relationship,6221,6222,Contains,2
1,1,relationship,6221,6223,Contains,2
2,2,relationship,6221,6224,Contains,2
3,3,relationship,6222,6225,Contains,2
4,4,relationship,6222,6226,Contains,2


In [56]:
# prompt: create a unique id for each row, make sure it is of type int64
#edges = edges.drop(columns=['id'])
import numpy as np
edges['id'] = np.arange(len(edges)).astype('int64')
edges.head()
edges['id'].dtype
edges['start'] = edges['start'].astype('int64')
edges['end'] = edges['end'].astype('int64')
edges['weight'] = edges['weight'].astype('int64')
edges.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48664 entries, 0 to 48663
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      48664 non-null  int64 
 1   type    48664 non-null  object
 2   start   48664 non-null  int64 
 3   end     48664 non-null  int64 
 4   label   48664 non-null  object
 5   weight  48664 non-null  int64 
dtypes: int64(4), object(2)
memory usage: 2.2+ MB


In [73]:
# prompt: selected_edges_t0 = edges[edges['end'].isin(tier0_node_ids)] and selected_edges_t0=edges[edges'start'] is not in tier0_node_ids

selected_edges_t0 = edges[(edges['end'].isin(tier0_node_ids)) & (~edges['start'].isin(tier0_node_ids))]

print(len(selected_edges_t0))
selected_nodes = nodes[nodes['id'].isin(tier2_node_ids)]
print(len(selected_nodes))

In [58]:
# print(nodes[nodes['id'] == 6371])
# print(nodes[nodes['id'] == 6263])
# print(nodes[nodes['id'] == 99999])
# print(nodes[nodes['id'] == 610])
# print(nodes[nodes['id'] == 457])
# print(nodes[nodes['id'] == 66])
# print(nodes[nodes['id'] == 36])
#print(nodes[nodes['id'] == 60])

# Dijkstra

In [60]:
import networkx as nx

# Create a directed graph
G = nx.DiGraph()

# Add nodes with their weights
for index, row in selected_nodes.iterrows():
    # Ensure node IDs are integers when adding to the graph
    G.add_node(int(row['id']), weight=row['weight'])

# Add edges with their weights
for index, row in selected_edges_t0.iterrows():
    # Ensure edge endpoints are integers when adding to the graph
    G.add_edge(row['start'], row['end'], weight=row['weight'])

# Compute shortest paths from each source node to all reachable nodes
shortest_paths = {}
for source in G.nodes():
    # Use single_source_dijkstra_path to find paths from one source to all others
    shortest_paths[source] = nx.single_source_dijkstra_path(G, source)

# Filter out paths from a node to itself and store in a new dictionary
shortest_paths_filtered = {
    source: {
        target: path
        for target, path in paths.items()
        if source != target
        #if target in tier0_node_ids
        #if source in tier2_node_ids
    }
    for source, paths in shortest_paths.items()
}
len(shortest_paths_filtered)

# Print the shortest paths (optional)
# for source, paths in shortest_paths_filtered.items():
#   if paths:
#     #print(f"Shortest paths from node {source}:")
#     for target, path in paths.items():
#         #print(f"  To node {target}: {path}")
#         ShortestPath = source, target, path
#        # print(ShortestPath)

8962

In [83]:
tier2_node_ids = pd.DataFrame(tier2_node_ids)
#

In [84]:
def get_edge_weight(start_node, end_node):
    # Find the corresponding edge in the 'edges' DataFrame
    edge = edges[(edges['start'] == start_node) & (edges['end'] == end_node)]
    if not edge.empty:
        # Assuming there is only one edge with this start and end for simplicity in path finding
        # If multiple edges exist, you might need to adjust based on which edge is part of the shortest path calculation
        return edge.iloc[0]['weight']

#     if not edge.empty:
# # Return the weight from the resulting DataFrame
#         return edge['weight'].values[0]
#     else:
#         return None

shortest_paths_list = []
for source, paths in shortest_paths_filtered.items():
    if paths:
        for target, path in paths.items():
#            get_edge_weight(source, target)
            shortest_paths_list.append({
                'source': int(source),
                'target': int(target),
                'path': path,
                'weight': int(get_edge_weight(source, target))
            })

#shortest_paths_list.info()
short = shortest_paths_list[shortest_paths_list['target'].isin(tier2_node_ids)]
#short1 = short[~short['source'].isin(tier0_node_ids)]
len(short)

In [79]:
shortest_paths_list = pd.DataFrame(shortest_paths_list)
shortest_paths_list.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   source  21 non-null     int64 
 1   target  21 non-null     int64 
 2   path    21 non-null     object
 3   weight  21 non-null     int64 
dtypes: int64(3), object(1)
memory usage: 804.0+ bytes


In [74]:
tier0_node_ids.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 343 entries, 0 to 342
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   0       343 non-null    int64
dtypes: int64(1)
memory usage: 2.8 KB


In [ ]:
display(edges[edges['start'] == 6222])

,id,type,start,end,label,weight
3,3,relationship,6222,6225,Contains,2
4,4,relationship,6222,6226,Contains,2
5,5,relationship,6222,6227,Contains,2


In [ ]:
import pandas as pd
#short= shortest_paths_list[shortest_paths_list['source'] isin tier2_node_ids]

#tier2_node_ids = pd.DataFrame(tier2_node_ids)

def get_edge_weight(start_node, end_node):
    # Find the corresponding edge in the 'edges' DataFrame
    edge = edges[(edges['start'] == start_node) & (edges['end'] == end_node)]
    # if not edge.empty:
    #     # Assuming there is only one edge with this start and end for simplicity in path finding
    #     # If multiple edges exist, you might need to adjust based on which edge is part of the shortest path calculation
    #     return edge.iloc[0]['weight']

    if not edge.empty:
# Return the weight from the resulting DataFrame
        return edge['weight'].values[0]
    else:
        return None

short = shortest_paths_list[shortest_paths_list['source'].isin(tier2_node_ids)]
short1 = short[short['target'].isin(tier0_node_ids)]
len(short1)
# max_value = short1['total_weight'].max()
# max_rows = short1[short1['total_weight'] == max_value]
# len(max_rows)
#df_filtered_paths.loc[df_filtered_paths['total_weight'].idxmin()]

# Calculate the total weight for each shortest path

shortest_paths_with_weight = []

# Iterate through the DataFrame 'short1' using 'iterrows()'
for index, path_info in short1.iterrows():
    total_weight = 0
# The path is expected to be a list of node IDs
    path = path_info['path']
    source = path_info['source']
    target = path_info['target']

# Ensure that 'path' is a list; you may need to adjust this if 'path' is a string
    if isinstance(path, list):
    # Iterate through the path to get the edges
        for i in range(len(path) - 1):
            start_node = path[i]
            end_node = path[i + 1]
        # Assuming you have a way to get the weight of the edge between start_node and end_node
        # For example, if you have a function or a mapping for edge weights
            edge_weight = get_edge_weight(start_node, end_node) # Define this function to retrieve the weight
            total_weight += edge_weight
            shortest_paths_with_weight.append({'source': source, 'target': target, 'path': path, 'total_weight': total_weight})

# Append the result (path and its total weight) to the list
#shortest_paths_with_weight.append({source: source, target: target, path: path, 'total_weight': total_weight})

df_shortest_paths_with_weight = pd.DataFrame(shortest_paths_with_weight)
# At this point, shortest_paths_with_weight will contain all paths and their corresponding weights
len(shortest_paths_with_weight)

0